# CRPS ensemble 방식

In [1]:
import numpy as np
import properscoring

# Example test set
observed_values = np.array([4.0, 1.0, 1.5, 5.5])  # Replace with your observed values
predictive_distributions = np.array([
    [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0],  # Replace with predictive distribution for the first observation
    [0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5],  # Replace with predictive distribution for the second observation
    [2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5],  # Replace with predictive distribution for the third observation
    [4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0]   # Replace with predictive distribution for the fourth observation
])

# Calculate CRPS for each observation
crps_values = [properscoring.crps_ensemble(observed_value, pred_dist) 
               for observed_value, pred_dist in zip(observed_values, predictive_distributions)]

# Calculate average CRPS for the entire test set
average_crps = np.mean(crps_values)

print("Average CRPS for the test set:", average_crps) # 낮을수록 좋은것

Average CRPS for the test set: 1.0714285714285714


In [2]:
crps_values # 4개 point에 대한 구간이(구간 개수 총 4개) 점수를 내었다. 

[np.float64(0.5714285714285714),
 np.float64(1.5),
 np.float64(1.9285714285714286),
 np.float64(0.2857142857142857)]

In [3]:
crps_values = [(observed_value, pred_dist)
               for observed_value, pred_dist in zip(observed_values, predictive_distributions)]

In [4]:
crps_values

[(np.float64(4.0), array([1., 2., 3., 4., 5., 6., 7.])),
 (np.float64(1.0), array([0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5])),
 (np.float64(1.5), array([2.5, 3. , 3.5, 4. , 4.5, 5. , 5.5])),
 (np.float64(5.5), array([4. , 4.5, 5. , 5.5, 6. , 6.5, 7. ]))]

# CRPS 민수씨 방식

In [5]:
# Example test set
observed_values = np.array([4.0, 1.0, 1.5, 5.5])  # Replace with your observed values
predictive_distributions = np.array([
    [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0],  # Replace with predictive distribution for the first observation
    [0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5],  # Replace with predictive distribution for the second observation
    [2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5],  # Replace with predictive distribution for the third observation
    [4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0]   # Replace with predictive distribution for the fourth observation
])

# 똑같이 얘네들이 들어왔을때, 
#[0.39999999999999997, 1.7, 2.1, 0.19999999999999998] 가 어떻게 바뀔까?

In [6]:
import pandas as pd
# 자체 제작 방식(7분위 pinball): 기준 τ 7개 분포에서 분위수 7개를 뽑아 점수를 낸다
LEVELS = np.array([0.1, 0.2, 0.3, 0.5, 0.7, 0.8, 0.9])

def pinball(q, y, tau):
    return np.where(y >= q, (y - q) * tau, (q - y) * (1 - tau))

def crps7(q7, y):                      # 예시라 var_std 정규화는 생략
    return 2 * np.mean([pinball(q7[k], y, LEVELS[k]) for k in range(7)])

In [7]:
my_values, q7_list = [], []
for observed_value, pred_dist in zip(observed_values, predictive_distributions):
    q7 = np.quantile(pred_dist, LEVELS)          # 기준 τ 7개 -> 분위수 7개 (선형 보간)
    q7_list.append(q7)
    my_values.append(crps7(q7, observed_value))
print("7분위 CRPS:", np.round(my_values, 3), " 평균:", round(np.mean(my_values), 3))
print("ensemble  :", np.round([properscoring.crps_ensemble(o, p) for o, p in zip(observed_values, predictive_distributions)], 3), " 평균:", round(float(np.mean([properscoring.crps_ensemble(o, p) for o, p in zip(observed_values, predictive_distributions)])), 3))

7분위 CRPS: [0.549 1.506 2.003 0.274]  평균: 1.083
ensemble  : [0.571 1.5   1.929 0.286]  평균: 1.071


In [8]:
# 첫 번째 예시: 기준 τ 7개 분포에서 뽑힌 분위수 7개와 pinball 계산 과정
y, q7 = observed_values[0], q7_list[0]
q_base = np.quantile(predictive_distributions[0], LEVELS, method="inverted_cdf")   # 베이스 코드(기준 앙상블)가 crps_ensemble에서 갖는 실제 분위수(계단 방식)
df_q = pd.DataFrame({"τ": LEVELS,
                     "베이스 코드 분위값": q_base,
                     "자체 제작 분위값": q7,
                     "pinball(베이스 코드)": [pinball(q_base[k], y, LEVELS[k]) for k in range(7)],
                     "pinball(자체 제작)": [pinball(q7[k], y, LEVELS[k]) for k in range(7)]})
display(df_q.style.format("{:.3f}").hide(axis="index"))

τ,베이스 코드 분위값,자체 제작 분위값,pinball(베이스 코드),pinball(자체 제작)
0.100,1.000,1.600,0.300,0.240
0.200,2.000,2.200,0.400,0.360
0.300,3.000,2.800,0.300,0.360
0.500,4.000,4.000,0.000,0.000
0.700,5.000,5.200,0.300,0.360
0.800,6.000,5.800,0.400,0.360
0.900,7.000,6.400,0.300,0.240


In [9]:
# 왜 값이 다른가: 분위수 수준 τ를 촘촘히(700개) 쓰면 ensemble 값과 같아진다
# (n은 기준 앙상블 개수 7의 배수여야 정확히 같아진다. 1000처럼 7의 배수가 아니면 구간 비중이 조금 어긋난다)
def crps_grid(sample, y, n=700):
    taus = (np.arange(n) + 0.5) / n
    qs = np.quantile(sample, taus, method="inverted_cdf")   # 표본의 경험분포 분위수
    return 2 * np.mean(np.where(y >= qs, (y - qs) * taus, (qs - y) * (1 - taus)))

display(pd.DataFrame({
    "ensemble(정확)": [properscoring.crps_ensemble(o, p) for o, p in zip(observed_values, predictive_distributions)],
    "촘촘한 τ(700개)": [crps_grid(p, o) for o, p in zip(observed_values, predictive_distributions)],
    "촘촘한 τ(1000개)": [crps_grid(p, o, n=1000) for o, p in zip(observed_values, predictive_distributions)],
    "7분위(자체 제작 방식, τ 7개)": my_values,
}).round(3))

,ensemble(정확),촘촘한 τ(700개),촘촘한 τ(1000개),"7분위(자체 제작 방식, τ 7개)"
0,0.571,0.571,0.573,0.549
1,1.500,1.500,1.500,1.506
2,1.929,1.929,1.928,2.003
3,0.286,0.286,0.286,0.274


# 7분위 값을 앙상블로 넣어 `crps_ensemble` 사용

기준 앙상블에서 뽑은 7분위 값 7개(`q7`, τ = 0.1, 0.2, 0.3, 0.5, 0.7, 0.8, 0.9)를 앙상블로 보고 `crps_ensemble`에 넣는다.
- 가중치 없음: 7개를 같은 비중으로 취급
- 가중치 있음: 이웃한 τ의 가운데를 경계로 한 구간 폭을 `weights`로 줌 (0.15, 0.10, 0.15, 0.20, 0.15, 0.10, 0.15)

가중치는 임의로 정한 것이라 근거가 약하고, 가중치 없는 쪽이 자체 제작 방식에 더 가깝다.

In [10]:
W7 = np.array([0.15, 0.10, 0.15, 0.20, 0.15, 0.10, 0.15])
ens7 = [properscoring.crps_ensemble(o, q) for o, q in zip(observed_values, q7_list)]
ens7w = [properscoring.crps_ensemble(o, q, weights=W7) for o, q in zip(observed_values, q7_list)]
display(pd.DataFrame({
    "ensemble(τ 7개)": [properscoring.crps_ensemble(o, p) for o, p in zip(observed_values, predictive_distributions)],
    "7분위(자체 제작 방식, τ 7개)": my_values,
    "7분위 값을 앙상블로(가중치 없음)": ens7,
    "7분위 값을 앙상블로(τ 구간 가중치)": ens7w,
}).round(3))

,ensemble(τ 7개),"7분위(자체 제작 방식, τ 7개)",7분위 값을 앙상블로(가중치 없음),7분위 값을 앙상블로(τ 구간 가중치)
0,0.571,0.549,0.563,0.486
1,1.500,1.506,1.520,1.546
2,1.929,2.003,2.010,2.023
3,0.286,0.274,0.282,0.243


### 정리
- `crps_ensemble`은 값 N개를 τ N개의 분위수로 보고 CRPS를 정확히 계산한다.
- 자체 제작 방식은 같은 정의(분위수별 pinball의 평균 x 2)를 τ 7개로만 근사하므로 값이 다르다. τ를 촘촘히 하면 ensemble 값으로 수렴한다.
- 7분위는 τ가 0.1~0.9에 있어 양 끝 10% 꼬리를 보지 않는다.